In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv
import http.client
import json
import requests
import os

C:\Users\UTTAM\AppData\Local\Temp\ipykernel_3316\2317543846.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
load_dotenv()
api = os.getenv('OPENAI_KEY')

In [4]:
@tool
def get_geo_location(place):
    """ this tool is used to fetch the geo location that is latitude and longitude of the given place"""
    conn = http.client.HTTPSConnection("address-from-to-latitude-longitude.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "75d3aaafbcmshecf4cc8f0487295p1f1c76jsna54a9fdef709",
        'x-rapidapi-host': "address-from-to-latitude-longitude.p.rapidapi.com",
        'Content-Type': "application/json"
    }

    conn.request("GET", f"""/geolocationapi?address={place}""", headers=headers)

    res = conn.getresponse()
    data = res.read()

    # Convert bytes to Python dictionary
    json_data = json.loads(data.decode("utf-8"))
    return json_data["Results"][0]
    

In [5]:
@tool
def get_weather_forecast(place):
    """This tool is use to fetch the current weather data for a given place"""
    response = requests.get(f"http://api.weatherapi.com/v1/current.json?key=21af37ed10e7477f98855731262807&q={place}&aqi=no")
    if response.status_code == 200:
        data = response.json()

        return {
            "place": data["location"]["name"],
            "condition": data["current"]["condition"]["text"],
            "temperature": data["current"]["temp_c"],
            "is_day": data["current"]["is_day"],
        }
    else:
        return {
            "error":'failed to fetch the data from the api'
        }
        


In [ ]:
# get_weather_forecast('patna')

In [6]:
search_tool = DuckDuckGoSearchRun()

In [7]:
llm = ChatOpenAI(
    model = 'gpt-4.1-nano',
    api_key = api,
    temperature = 0
)

In [8]:
agent = create_agent(
    model = llm,
    tools =[search_tool, get_weather_forecast]
)

In [11]:
messages = []
query = HumanMessage(content ='give me 5 places in uttar pradesh to visit, where the wether is sunny')
messages.append(query)

In [12]:
response = agent.invoke({'messages':messages})
print(response["messages"][-1].content)

Based on the search results, the best time to visit Uttar Pradesh for sunny weather is during the winter months, from October to March. Here are five places in Uttar Pradesh that you can visit when the weather is sunny and pleasant:

1. Ayodhya: A holy city on the banks of the Sarayu River, famous for its religious significance and Lord Ram's birthplace.
2. Lucknow: The capital city known for its historical monuments, palaces, and vibrant culture.
3. Allahabad (Prayagraj): Renowned for the confluence of rivers and religious festivals, offering a mix of spirituality and history.
4. Varanasi: A spiritual city on the banks of the Ganges, famous for its ghats, temples, and cultural heritage.
5. Kanpur: An industrial city with historical sites, parks, and a lively atmosphere.

Would you like more detailed information about any of these places?
